## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Pose estimation - Application, part 1: the Sit-to-Stand test

In this notebook, we use a pose estimation model to analyze the **Sit-to-Stand (STS) test**.

## STS Test

In the five-repetition STS test, a person sits on a chair with the arms crossed, and stands up and sits down **five times** as fast as possible. It is one of the simplest and most informative functional tests of mobility:
- lower-limb strength
- balance and postural control
- coordination
- movement speed (vigor)

Clinically, STS performance is used to assess:
- aging-related decline
- musculoskeletal conditions (e.g., osteoarthritis)
- neurological impairments

## Dataset
>Boswell, M. A., Kidziński, Ł., Hicks, J. L., Uhlrich, S. D., Falisse, A., & Delp, S. L. (2023). Smartphone videos of the sit-to-stand test predict osteoarthritis and health outcomes in a nationwide study. *npj Digital Medicine*, 6(1), 32.

The authors showed that smartphone videos recorded at home can give useful biomechanical measures. Their pipeline:
1. Record a short video of a person who does the five-repetition STS test (the camera is at about 45 degrees to the person)
2. Use pose estimation to find the body keypoints
3. Convert the keypoints into movement signals and measures (for example the total time, and the **trunk angle**)
4. Relate those measures to clinical outcomes

Their main finding: people with hip or knee osteoarthritis **lean the trunk forward more** when they stand up, even after controlling for age, sex and BMI.

The authors made part of the videos available (89 videos), with age, sex, BMI and osteoarthritis status. They are in `data/sit2stand`.

<div style="text-align:center">
<img src="images/Sit2Stand.jpg" alt="plot" width="800" />
</div>

In this notebook, we analyze **one video** and decide how to extract useful measures:
1. run pose estimation
2. make a movement signal that does not depend on the distance to the camera
3. detect the repetitions
4. compute the speed and the trunk angle
5. compute a set of measures (features)

## Part A - Pose estimation

In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO
from scipy.signal import savgol_filter, find_peaks, medfilt
from ipywidgets import IntSlider, Play, HBox, VBox, jslink, interactive_output
from IPython.display import display

edges = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
         (11, 13), (13, 15), (12, 14), (14, 16), (0, 5), (0, 6)]
keypoint_labels = {0: 'nose', 1: 'left_eye', 2: 'right_eye', 3: 'left_ear', 4: 'right_ear',
                   5: 'left_shoulder', 6: 'right_shoulder', 7: 'left_elbow', 8: 'right_elbow',
                   9: 'left_wrist', 10: 'right_wrist', 11: 'left_hip', 12: 'right_hip',
                   13: 'left_knee', 14: 'right_knee', 15: 'left_ankle', 16: 'right_ankle'}

We run the YOLO pose model on every frame (about 1 to 2 minutes on a CPU). As in 08.01:
- we keep the person with the **largest box**,
- we replace keypoints with a confidence below 0.3 by NaN.

In [ ]:
video_path = "data/sit2stand/videos_sit2stand_valid/4RF13po6.mp4"
model = YOLO("../models/yolo26s-pose.pt")

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Frame rate: {fps} frames/s | size: {w} x {h}')

pose_seq = []
for r in model.predict(source=video_path, stream=True, verbose=False):
    kp = np.full((17, 2), np.nan)
    if r.boxes is not None and len(r.boxes) > 0 and r.keypoints is not None:
        b = r.boxes.xyxy.cpu().numpy()
        k = int(np.argmax((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])))        # largest box
        conf = r.keypoints.conf.cpu().numpy()[k]
        kp = np.where(conf[:, None] >= 0.3, r.keypoints.xy.cpu().numpy()[k], np.nan)
    pose_seq.append(kp)
pose_seq = np.array(pose_seq)               # (frames, 17, 2)
t = np.arange(len(pose_seq)) / fps          # time of each frame, in seconds
print('Frames:', pose_seq.shape[0], '| duration:', round(t[-1], 1), 's')

In [ ]:
def plot_pose_frame(frame_idx):
    k = pose_seq[frame_idx]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(k[:, 0], k[:, 1], s=30, c="red")
    for a, b in edges:
        ax.plot(k[[a, b], 0], k[[a, b], 1], color="tab:cyan", linewidth=2)
    ax.set_xlim(0, w); ax.set_ylim(h, 0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"Frame {frame_idx} ({t[frame_idx]:.1f} s)")
    plt.show()

slider = IntSlider(value=0, min=0, max=len(pose_seq) - 1, step=1, description="Frame")
play = Play(value=0, min=0, max=len(pose_seq) - 1, step=1, interval=50, description="Press play")
jslink((play, "value"), (slider, "value"))
display(VBox([HBox([play, slider]), interactive_output(plot_pose_frame, {"frame_idx": slider})]))

## Part A.1 - A movement signal

The feet stay on the ground during the test, so we can use the **vertical distance between the hips and the ankles**: it is small when the person sits and large when the person stands.

Two problems:
1. **Missing keypoints.** Some frames have NaN values. We fill short gaps by interpolation.
2. **Noise.** Pose estimation is noisy from frame to frame. We use a median filter (lecture 03.03), which removes single-frame spikes, then a Savitzky–Golay filter to smooth the signal.

In [ ]:
def fill_gaps(x):
    """Replace NaN values by linear interpolation."""
    x = np.asarray(x, dtype=float).copy()
    ok = np.isfinite(x)
    x[~ok] = np.interp(np.flatnonzero(~ok), np.flatnonzero(ok), x[ok])
    return x

def smooth(x):
    return savgol_filter(medfilt(fill_gaps(x), 5), window_length=11, polyorder=2)

hip = np.nanmean(pose_seq[:, [11, 12], :], axis=1)       # mid-hip (x, y)
knee = np.nanmean(pose_seq[:, [13, 14], :], axis=1)
ankle = np.nanmean(pose_seq[:, [15, 16], :], axis=1)
shoulder = np.nanmean(pose_seq[:, [5, 6], :], axis=1)

hip_above_ankle_px = smooth(ankle[:, 1] - hip[:, 1])      # y increases downwards, so ankle - hip > 0

plt.figure(figsize=(10, 4))
plt.plot(t, hip_above_ankle_px)
plt.xlabel("Time (s)"); plt.ylabel("Hip height above ankle (pixels)")
plt.title("Sit-to-stand movement")
plt.show()

We can see the five repetitions: the signal goes up when the person stands, and down when the person sits.

However, there is a **big problem**: the signal is in pixels. It depends on the distance from the person to the camera and on the image resolution. We cannot compare two videos.

## Part A.2 - Normalization

We usually do not have a true length in a video. So we divide by a **body length** that does not change during the movement. Here we use the **leg length**: thigh (hip–knee) + shank (knee–ankle), as the median over all frames. The result has no units: 1.0 means one leg length.

(The box height is a poor choice: it changes when the person stands or sits, or raises the arms.)

In [ ]:
thigh = np.linalg.norm(hip - knee, axis=1)
shank = np.linalg.norm(knee - ankle, axis=1)
leg_length = np.nanmedian(thigh + shank)
print(f'Leg length: {leg_length:.0f} pixels')

height = hip_above_ankle_px / leg_length              # in leg lengths
plt.figure(figsize=(10, 4))
plt.plot(t, height)
plt.xlabel("Time (s)"); plt.ylabel("Hip height above ankle (leg lengths)")
plt.title("Sit-to-stand movement (normalized)")
plt.show()

## Part A.3 - Detect the repetitions

Each standing position is a **peak** of the signal. We use `find_peaks` with:
- a minimum distance of 0.8 s between two peaks,
- a minimum prominence of 40% of the range of the signal (small bumps are not repetitions).

In [ ]:
signal_range = np.percentile(height, 95) - np.percentile(height, 5)
stand_peaks, _ = find_peaks(height, distance=int(0.8 * fps), prominence=0.4 * signal_range)
print('Number of repetitions:', len(stand_peaks))

plt.figure(figsize=(10, 4))
plt.plot(t, height)
plt.plot(t[stand_peaks], height[stand_peaks], "x", color="green", markersize=10, label="Standing")
plt.xlabel("Time (s)"); plt.ylabel("Hip height (leg lengths)")
plt.legend()
plt.show()

## Part A.4 - Speed

The derivative of the height gives the vertical **speed** of the hips (in leg lengths per second). We compute it with the Savitzky–Golay filter (`deriv=1`, `delta=1/fps`). Positive peaks: maximum speed when standing up. Negative peaks: maximum speed when sitting down.

In [ ]:
speed = savgol_filter(height, window_length=11, polyorder=2, deriv=1, delta=1 / fps)

up_peaks, _ = find_peaks(speed, distance=int(0.5 * fps), prominence=0.3 * np.max(speed))
down_peaks, _ = find_peaks(-speed, distance=int(0.5 * fps), prominence=0.3 * np.max(-speed))

plt.figure(figsize=(10, 4))
plt.plot(t, speed, color='tab:orange')
plt.plot(t[up_peaks], speed[up_peaks], "x", color="green", label="Max speed standing up")
plt.plot(t[down_peaks], speed[down_peaks], "x", color="red", label="Max speed sitting down")
plt.axhline(0, color='gray', linewidth=0.8)
plt.xlabel("Time (s)"); plt.ylabel("Vertical speed (leg lengths/s)")
plt.legend()
plt.show()

## Part A.5 - Trunk angle

Boswell et al. found that the **trunk angle** is related to osteoarthritis. We compute the angle between the trunk (the line from the mid-hip to the mid-shoulder) and the vertical: 0 degrees = upright, larger = leaning forward.

From the angle we also compute the **angular speed** and the **angular acceleration** of the trunk (the first and second derivatives).

Note: this is a 2D angle in the image. The camera is at about 45 degrees, so the angle mixes forward lean and side lean. It is not the true 3D trunk angle.

In [ ]:
trunk_angle = np.degrees(np.arctan2(np.abs(shoulder[:, 0] - hip[:, 0]), hip[:, 1] - shoulder[:, 1]))
trunk_angle = smooth(trunk_angle)
trunk_speed = savgol_filter(trunk_angle, 11, 2, deriv=1, delta=1 / fps)      # deg/s
trunk_acc = savgol_filter(trunk_angle, 11, 2, deriv=2, delta=1 / fps)        # deg/s^2

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t, height); ax[0].set_ylabel('Hip height (leg lengths)')
ax[1].plot(t, trunk_angle, color='tab:purple'); ax[1].set_ylabel('Trunk angle (deg)')
ax[1].set_xlabel('Time (s)')
plt.show()

The trunk leans forward **two times** in each repetition: just before the person stands up (to move the body weight over the feet), and again when the person sits down. When the person stands, the trunk is upright (small angle).

## Part A.6 - Features

Now we compute a small set of measures for this video:

| Feature | Meaning |
| --- | --- |
| `sts_time` | time from the start of the first stand-up to the end of the fifth sit-down (s) |
| `rep_time` | median time between two standing positions (s) |
| `stand_speed`, `sit_speed` | median of the maximum speed standing up / sitting down (leg lengths/s) |
| `trunk_angle_max` | median of the maximum trunk angle before each stand-up (deg) |
| `trunk_speed_max`, `trunk_acc_max` | 95th percentile of the trunk angular speed (deg/s) and acceleration (deg/s²) |

In [ ]:
def sts_features(height, trunk_angle, fps):
    speed = savgol_filter(height, 11, 2, deriv=1, delta=1 / fps)
    trunk_speed = savgol_filter(trunk_angle, 11, 2, deriv=1, delta=1 / fps)
    trunk_acc = savgol_filter(trunk_angle, 11, 2, deriv=2, delta=1 / fps)
    rng = np.percentile(height, 95) - np.percentile(height, 5)
    stand = find_peaks(height, distance=int(0.8 * fps), prominence=0.4 * rng)[0]
    up = find_peaks(speed, distance=int(0.5 * fps), prominence=0.3 * np.max(speed))[0]
    down = find_peaks(-speed, distance=int(0.5 * fps), prominence=0.3 * np.max(-speed))[0]
    # total time: from the last "sitting" frame before the first stand to the first "sitting" frame after the fifth stand
    sitting_level = np.percentile(height, 5) + 0.1 * rng
    sts_time = np.nan
    if len(stand) >= 5:
        before = np.flatnonzero(height[:stand[0]] < sitting_level)
        after = np.flatnonzero(height[stand[4]:] < sitting_level)
        start = before[-1] if len(before) else 0
        end = stand[4] + after[0] if len(after) else len(height) - 1
        sts_time = (end - start) / fps
    trunk_max = [np.max(trunk_angle[max(0, p - int(fps)):p + 1]) for p in stand]
    return {'n_reps': len(stand), 'sts_time': sts_time,
            'rep_time': np.median(np.diff(stand)) / fps if len(stand) > 1 else np.nan,
            'stand_speed': np.median(speed[up]) if len(up) else np.nan,
            'sit_speed': np.median(-speed[down]) if len(down) else np.nan,
            'trunk_angle_max': np.median(trunk_max) if len(trunk_max) else np.nan,
            'trunk_speed_max': np.percentile(np.abs(trunk_speed), 95),
            'trunk_acc_max': np.percentile(np.abs(trunk_acc), 95)}

pd.Series(sts_features(height, trunk_angle, fps)).round(2)

## What happens with all 89 videos?

We applied the same pipeline to the 89 videos (this takes about 30 minutes, so we do not run it here). Then we tested, with repeated cross-validation, if the measures can detect osteoarthritis (23 of the 89 participants):

| Inputs | AUC (mean ± SD) |
| --- | --- |
| Age, sex and BMI | 0.57 ± 0.15 |
| Trunk angular acceleration | 0.67 ± 0.14 |

Adjusted for age, sex and BMI, a larger trunk acceleration was related to osteoarthritis (odds ratio about 2 per standard deviation). This agrees with the paper. However:
- the effect is **modest**, and the spread between folds is large: 89 people are not many;
- other measures (total time, speed) did not add information to age, sex and BMI;
- we tried several measures and several targets before we found this result. With small data, this increases the chance of a false positive. A new dataset would be needed to confirm it.

To see what pose estimation can do with **more data**, part 2 uses a dataset of more than 3,000 clinical gait videos.